# 02 Cleaning and join

**What this notebook does.** It diagnoses the training table, rebuilds it as a continuous daily grid per store and item, cleans the dimension tables, builds a store-day calendar and flags days without records. The cells depend on each other and must be run from top to bottom. The decisions are summarised in the data preparation log at the end.

## 1. Diagnose the training table

**Purpose.** Find the data quality problems of `train` (cells 1 to 3) before deciding how to clean it. Nothing is changed in these cells.

**Method.** The Parquet files from notebook 01 are loaded (2.4 GB in memory). The cells count missing dates, missing promotion values, negative sales, zero-sales rows, duplicate keys, non-integer sales by family, extreme values and the number of store-item pairs.

**Findings.**
- 125,497,040 rows, 54 stores, 4,036 items and 1,684 days. 25 December is missing in 2013 to 2016 (also in `transactions`, so stores were closed).
- The key (date, store, item) has no duplicates, and there are no rows with zero sales: a row exists only when something was sold, so the data are sparse.
- `onpromotion` is missing for the first 15 months (January 2013 to March 2014) and recorded only on rows with sales.
- 7,795 negative sales (0.006%; median -1, minimum -15,372).
- 6.53% of sales are not whole numbers (poultry 99.7%, meats 98.9%, produce 48%, seafood 34%): these are sold by weight.
- Extreme values exist (for example meats: 99.9th percentile 364, maximum 89,440).
- 174,685 store-item pairs were sold at least once.

**Limitation.** The diagnostics describe the full data but several rules (for example thresholds) are decided in later cells.

In [11]:
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
train = pd.read_parquet(BASE / "interim" / "train").drop(columns="id")
print("shape:", train.shape, "| RAM GB:", round(train.memory_usage(deep=True).sum() / 1e9, 2))

all_days = pd.date_range(train["date"].min(), train["date"].max())
present = pd.DatetimeIndex(train["date"].drop_duplicates())
missing = all_days.difference(present)
print("\ncalendar days:", len(all_days), "| days in data:", len(present))
print("missing dates:", [d.strftime("%Y-%m-%d") for d in missing])

ym = train["date"].dt.year * 100 + train["date"].dt.month
null_by_month = train["onpromotion"].isna().groupby(ym).mean()
print("\nmonths with null promo:", (null_by_month > 0).sum())
print(null_by_month[null_by_month > 0].round(3).tail(8))

neg = train.loc[train["unit_sales"] < 0, "unit_sales"]
print("\nnegative sales:", len(neg))
print(neg.describe().round(2))

print("\nzero-sales rows:", int((train["unit_sales"] == 0).sum()))
print("stores:", train["store_nbr"].nunique(), "| items:", train["item_nbr"].nunique())
print("duplicate (date, store, item):", int(train.duplicated(subset=["date", "store_nbr", "item_nbr"]).sum()))

shape: (125497040, 5) | RAM GB: 2.38

calendar days: 1688 | days in data: 1684
missing dates: ['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25']

months with null promo: 15
date
201308    1.0
201309    1.0
201310    1.0
201311    1.0
201312    1.0
201401    1.0
201402    1.0
201403    1.0
Name: onpromotion, dtype: float64

negative sales: 7795
count     7795.00
mean       -18.42
std        242.25
min     -15372.00
25%         -4.00
50%         -1.00
75%         -1.00
max         -0.00
Name: unit_sales, dtype: float64

zero-sales rows: 0
stores: 54 | items: 4036
duplicate (date, store, item): 0


In [12]:
items = pd.read_csv(BASE / "raw" / "items.csv")
print("items in items.csv:", len(items), "| items in train:", train["item_nbr"].nunique())

neg_rows = train[train["unit_sales"] < 0].merge(
    items[["item_nbr", "family", "perishable"]], on="item_nbr", how="left")
print("\n10 most negative rows:")
print(neg_rows.nsmallest(10, "unit_sales"))
print("\nnegative by family (top 5):")
print(neg_rows["family"].value_counts().head(5))
print("tiny negatives (> -1):", int((neg_rows["unit_sales"] > -1).sum()))

frac = train["unit_sales"] % 1 != 0
print("\nnon-integer unit_sales rows:", int(frac.sum()), f"({frac.mean():.2%})")

END = train["date"].max()
first = train.groupby(["store_nbr", "item_nbr"])["date"].min()
active_days = (END - first).dt.days + 1
print("\nstore-item pairs:", len(first))
print("rows if filled from first sale to end:", int(active_days.sum()))
print("rows if full grid:", 54 * train["item_nbr"].nunique() * 1684)

items in items.csv: 4100 | items in train: 4036

10 most negative rows:
           date  store_nbr  item_nbr    unit_sales  onpromotion  \
2910 2015-06-22         18   1166474 -15372.000000        False   
6805 2017-04-06         32   1158720 -10002.000000        False   
4787 2016-05-14         53    119026  -4673.000000        False   
4788 2016-05-14         53    323921  -3606.000000        False   
4794 2016-05-14         53   1229028  -3600.000000        False   
6074 2016-12-20         38   2010755  -3451.363037        False   
4622 2016-04-24         46   1463852  -2487.000000        False   
3037 2015-07-20         49   1082042  -2400.000000        False   
4805 2016-05-16         29    812769  -2400.000000        False   
6810 2017-04-07          7   1430040  -1943.000000        False   

                family  perishable  
2910         BEVERAGES           0  
6805         GROCERY I           0  
4787          CLEANING           0  
4788     PERSONAL CARE           0  
4794 

In [13]:
items = pd.read_csv(BASE / "raw" / "items.csv")
tx = pd.read_csv(BASE / "raw" / "transactions.csv", parse_dates=["date"])

fam = items.set_index("item_nbr")["family"].astype("category")
train["family"] = train["item_nbr"].map(fam)

frac = train["unit_sales"] % 1 != 0
print("share of non-integer sales by family:")
print(frac.groupby(train["family"], observed=True).mean().sort_values(ascending=False).head(6).round(3))

g = train.groupby("family", observed=True)["unit_sales"]
print("\nupper tail by family:")
print(pd.DataFrame({"p99.9": g.quantile(0.999), "max": g.max()}).sort_values("max", ascending=False).head(6).round(0))

print("\ntransactions rows on 12-25:", int((tx["date"].dt.strftime("%m-%d") == "12-25").sum()))
span = train.groupby("store_nbr")["date"].agg(first="min", last="max")
span["sales_days"] = train.groupby("store_nbr")["date"].nunique()
span["tx_days"] = tx.groupby("store_nbr")["date"].nunique()
print("\nstores whose first sale is after 2013-01-02:", int((span["first"] > "2013-01-02").sum()))
print(span.sort_values("first").tail(8))

share of non-integer sales by family:
family
POULTRY           0.997
MEATS             0.989
PRODUCE           0.480
SEAFOOD           0.342
DELI              0.082
PREPARED FOODS    0.063
Name: unit_sales, dtype: float64

upper tail by family:
              p99.9      max
family                      
MEATS         364.0  89440.0
GROCERY I     137.0  44142.0
BEVERAGES     342.0  16000.0
FROZEN FOODS  471.0  12021.0
POULTRY       265.0  12016.0
CLEANING       87.0  10685.0

transactions rows on 12-25: 0

stores whose first sale is after 2013-01-02: 8
               first       last  sales_days  tx_days
store_nbr                                           
36        2013-05-09 2017-08-15        1553     1551
53        2014-05-29 2017-08-15        1169     1167
20        2015-02-13 2017-08-15         911      909
29        2015-03-20 2017-08-15         876      874
21        2015-07-24 2017-08-15         750      748
42        2015-08-21 2017-08-15         722      720
22        2015-10-09

## 2. Build the daily sales grid

**Purpose.** Lags, rolling statistics and ADI need a continuous daily series, but the data contain rows only for days with a sale. The grid adds the missing days.

**Method.** For each store-item pair a row is added for every day from its first sale to the end of the data, using only the 1,684 days present in the data (so 25 December is not invented). Added rows get `unit_sales = 0` and an unknown promotion value (-1). `unit_sales_raw` keeps the original value, `unit_sales` is clipped at zero and `is_return` marks negative values. Checks stop the cell if the number of non-zero grid rows or the total sales differ from the original table. The next cell confirms the saved row count.

**Output.** 174,685 store-item pairs and 194,660,451 rows (69,163,411 added zeros, 35.5%), saved in `clean/sales_grid/` partitioned by year.

**Limitation.** The added zeros are assumed zero-sales days. They can mean no demand, a stock-out, a product not on display, a discontinued product (series keep receiving zeros until the end of the data) or a missing record.

In [14]:
import numpy as np
import shutil

train = train.drop(columns="family", errors="ignore")

dates = np.sort(train["date"].unique())
n_days = len(dates)
date_idx = pd.Index(dates).get_indexer(train["date"])

key = train["store_nbr"].astype("int64") * 10_000_000 + train["item_nbr"].astype("int64")
codes, uniques = pd.factorize(key, sort=True)
store_of = (uniques // 10_000_000).astype("int8")
item_of = (uniques % 10_000_000).astype("int32")

first_idx = pd.Series(date_idx).groupby(codes).min().to_numpy().astype("int64")
counts = n_days - first_idx
offsets = np.cumsum(counts) - counts
total = int(counts.sum())
print("store-item pairs:", f"{len(uniques):,}", "| grid rows:", f"{total:,}")

pos = offsets[codes] + date_idx - first_idx[codes]

raw = np.zeros(total, dtype="float32")
raw[pos] = train["unit_sales"].to_numpy()

p = train["onpromotion"].to_numpy(dtype="float32", na_value=np.nan)
promo = np.full(total, -1, dtype="int8")
promo[pos] = np.nan_to_num(p, nan=-1).astype("int8")

grid_date_idx = (np.arange(total, dtype="int64")
                 - np.repeat(offsets, counts)
                 + np.repeat(first_idx, counts)).astype("int16")

grid = pd.DataFrame({
    "date": pd.to_datetime(dates)[grid_date_idx],
    "store_nbr": np.repeat(store_of, counts),
    "item_nbr": np.repeat(item_of, counts),
    "unit_sales_raw": raw,
    "unit_sales": np.clip(raw, 0, None),
    "is_return": raw < 0,
    "onpromotion": promo,
})

assert int((raw != 0).sum()) == len(train)
assert np.isclose(raw.sum(dtype="float64"), train["unit_sales"].to_numpy().sum(dtype="float64"))
print(f"rows: {len(train):,} -> {len(grid):,} (+{len(grid) - len(train):,} filled zeros)")

OUT = BASE / "clean" / "sales_grid"
shutil.rmtree(OUT, ignore_errors=True)
grid["year"] = grid["date"].dt.year.astype("int16")
grid.to_parquet(OUT, partition_cols=["year"], index=False)
print("saved to", OUT)

store-item pairs: 174,685 | grid rows: 194,660,451
rows: 125,497,040 -> 194,660,451 (+69,163,411 filled zeros)
saved to /Users/tinnguyen/Projects/bigdata/clean/sales_grid


In [15]:
import pyarrow.dataset as ds
d = ds.dataset(BASE / "clean" / "sales_grid", format="parquet", partitioning="hive")
n_saved = d.count_rows()
print(f"{n_saved:,}")
assert n_saved == total, "saved grid does not have the expected number of rows"

194,660,451


## 3. Clean the dimension tables

**Purpose.** Prepare the small tables for joining.

**Method.** `oil`: reindexed to the 1,684 sales days; 521 days have no price (481 weekend days and 40 other days) and are forward-filled with the last known price, with the first day left missing and an `oil_imputed` flag. `items`: only the 4,036 items that were sold are kept (64 dropped). `stores` and `transactions`: compact data types. `holidays_events`: profiled here and processed in the next section. This cell needs `dates` and `item_of` from the grid cell.

**Output.** `oil`, `items`, `stores` and `transactions` as Parquet files in `clean/dim/`.

**Limitation.** Forward-filled prices are the last known price, not the true price of weekends and holidays.

In [16]:
RAW = BASE / "raw"
DIM = BASE / "clean" / "dim"
DIM.mkdir(parents=True, exist_ok=True)
days = pd.to_datetime(dates)

oil = pd.read_csv(RAW / "oil.csv", parse_dates=["date"]).set_index("date")["dcoilwtico"]
print("oil rows:", len(oil), "| null:", int(oil.isna().sum()))
oil = oil.reindex(days)
print("oil after reindex to", len(days), "sales days | missing:", int(oil.isna().sum()))
oil_df = pd.DataFrame({"oil_price": oil.ffill(), "oil_imputed": oil.isna()}).rename_axis("date").reset_index()
oil_df.to_parquet(DIM / "oil.parquet", index=False)

items = pd.read_csv(RAW / "items.csv")
n0 = len(items)
items = items[items["item_nbr"].isin(np.unique(item_of))].copy()
items["family"] = items["family"].astype("category")
items["class"] = items["class"].astype("int16")
items["perishable"] = items["perishable"].astype("int8")
print("items:", n0, "->", len(items))
items.to_parquet(DIM / "items.parquet", index=False)

stores = pd.read_csv(RAW / "stores.csv")
for c in stores.columns:
    if stores[c].dtype == "object":
        stores[c] = stores[c].astype("category")
stores["store_nbr"] = stores["store_nbr"].astype("int8")
print(stores.nunique())
stores.to_parquet(DIM / "stores.parquet", index=False)

tx = pd.read_csv(RAW / "transactions.csv", parse_dates=["date"])
tx["store_nbr"] = tx["store_nbr"].astype("int8")
tx["transactions"] = tx["transactions"].astype("int32")
tx.to_parquet(DIM / "transactions.parquet", index=False)

hol = pd.read_csv(RAW / "holidays_events.csv", parse_dates=["date"])
print("\n", hol["type"].value_counts().to_string())
print("\n", hol["locale"].value_counts().to_string())
print("\ntransferred:", hol["transferred"].value_counts().to_dict())
print("dates with more than one event:", int(hol["date"].duplicated().sum()))
print(hol[hol["transferred"]].head(5))

oil rows: 1218 | null: 43
oil after reindex to 1684 sales days | missing: 521
items: 4100 -> 4036
store_nbr    54
city         22
state        16
type          5
cluster      17
dtype: int64

 type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5

 locale
National    174
Local       152
Regional     24

transferred: {False: 338, True: 12}
dates with more than one event: 38
          date     type    locale locale_name                 description  \
19  2012-10-09  Holiday  National     Ecuador  Independencia de Guayaquil   
72  2013-10-09  Holiday  National     Ecuador  Independencia de Guayaquil   
135 2014-10-09  Holiday  National     Ecuador  Independencia de Guayaquil   
255 2016-05-24  Holiday  National     Ecuador        Batalla de Pichincha   
266 2016-07-25  Holiday     Local   Guayaquil      Fundacion de Guayaquil   

     transferred  
19          True  
72          True  
135         True  
255         True  
266    

## 4. Build the store-day calendar

**Purpose.** One table with the context of every store and day, joined once instead of being repeated in the 194.7M-row grid.

**Method.** One row per date and store (54 × 1,684 = 90,936) with store attributes, oil price, transactions and flags. Holiday flags use only rows of type Holiday, Transfer, Additional or Bridge that were not transferred (12 transferred rows are excluded) and are set by scope: national, regional (state) and local (city). Dates with more than one event (38) are combined into flags instead of duplicating rows. Other flags: `is_workday_swap`, `is_event`, `earthquake` (event description contains "Terremoto") and `is_payday` (the 15th and the last day of the month, an assumption about public salary payments that should be verified).

**Output.** `clean/dim/calendar_store.parquet`.

**Limitation.** `transactions` is known only after the day ends, so it is for analysis only. `earthquake` and some events cannot be known in advance, so they are not inputs for forecasting models.

In [17]:
RAW = BASE / "raw"
DIM = BASE / "clean" / "dim"

stores = pd.read_parquet(DIM / "stores.parquet")
oil_df = pd.read_parquet(DIM / "oil.parquet")
tx = pd.read_parquet(DIM / "transactions.parquet")
hol = pd.read_csv(RAW / "holidays_events.csv", parse_dates=["date"])

oil_raw = pd.read_csv(RAW / "oil.csv", parse_dates=["date"]).set_index("date")["dcoilwtico"]
absent = oil_df.loc[~oil_df["date"].isin(oil_raw.index), "date"]
print("days absent from oil.csv:", len(absent), "| weekends:", int((absent.dt.dayofweek >= 5).sum()))

cal = pd.MultiIndex.from_product([oil_df["date"], stores["store_nbr"]],
                                 names=["date", "store_nbr"]).to_frame(index=False)
cal = cal.merge(stores[["store_nbr", "city", "state", "type", "cluster"]]
                .rename(columns={"type": "store_type"}), on="store_nbr")
cal = cal.merge(oil_df, on="date", how="left")
cal = cal.merge(tx, on=["date", "store_nbr"], how="left")

real = hol[~hol["transferred"]]
off = real[real["type"].isin(["Holiday", "Transfer", "Additional", "Bridge"])]
nat = set(off.loc[off["locale"] == "National", "date"])
reg = set(zip(off.loc[off["locale"] == "Regional", "date"], off.loc[off["locale"] == "Regional", "locale_name"]))
loc = set(zip(off.loc[off["locale"] == "Local", "date"], off.loc[off["locale"] == "Local", "locale_name"]))

cal["hol_national"] = cal["date"].isin(nat)
cal["hol_regional"] = [(d, s) in reg for d, s in zip(cal["date"], cal["state"])]
cal["hol_local"] = [(d, c) in loc for d, c in zip(cal["date"], cal["city"])]
cal["is_holiday"] = cal[["hol_national", "hol_regional", "hol_local"]].any(axis=1)
cal["is_workday_swap"] = cal["date"].isin(set(real.loc[real["type"] == "Work Day", "date"]))
cal["is_event"] = cal["date"].isin(set(real.loc[real["type"] == "Event", "date"]))
cal["earthquake"] = cal["date"].isin(set(hol.loc[hol["description"].str.contains("Terremoto", case=False), "date"]))

d = cal["date"].dt
cal["dow"] = d.dayofweek.astype("int8")
cal["month"] = d.month.astype("int8")
cal["year"] = d.year.astype("int16")
cal["is_payday"] = (d.day == 15) | d.is_month_end

print("local holiday cities without a store:", sorted(set(off.loc[off["locale"] == "Local", "locale_name"]) - set(stores["city"])))
print("regional states without a store:", sorted(set(off.loc[off["locale"] == "Regional", "locale_name"]) - set(stores["state"])))

cal.to_parquet(DIM / "calendar_store.parquet", index=False)
flags = ["hol_national", "hol_regional", "hol_local", "is_holiday", "is_workday_swap", "is_event", "earthquake", "is_payday"]
print("\ncalendar_store:", cal.shape)
print("store-days flagged:\n", cal[flags].sum().to_string())
print("distinct dates flagged:\n", cal.groupby("date")[flags].max().sum().to_string())
print("store-days without transactions:", int(cal["transactions"].isna().sum()))

days absent from oil.csv: 481 | weekends: 481
local holiday cities without a store: []
regional states without a store: []

calendar_store: (90936, 20)
store-days flagged:
 hol_national       4212
hol_regional         31
hol_local           360
is_holiday         4598
is_workday_swap     270
is_event           2970
earthquake         1674
is_payday          5994
distinct dates flagged:
 hol_national        78
hol_regional        18
hol_local          104
is_holiday         190
is_workday_swap      5
is_event            55
earthquake          31
is_payday          111
store-days without transactions: 7448


## 5. Store status flags

**Purpose.** Separate days before a store started selling, days with a suspected closure and gaps in the transactions table.

**Method.** `store_open_date` is the first date with a sales row (a proxy for the opening date, not an official date) and `is_open` means the date is on or after it. `has_sales` marks store-days with at least one sales row. `is_closed` marks open store-days without any sales row (a suspected closure, examined below). `tx_gap` marks days with sales but no transactions row. `is_event_other` marks events other than the earthquake.

**Findings.** 7,448 store-days have no transactions row: 6,729 are before the store's first sale, 601 are open days without sales or transactions and 118 are days with sales but no transactions row (mostly 2 to 4 January 2016).

**Limitation.** The flags describe what is observed in the data; they do not prove that a store was open or closed.

In [18]:
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
DIM = BASE / "clean" / "dim"

cal = pd.read_parquet(DIM / "calendar_store.parquet")
sales = pd.read_parquet(BASE / "interim" / "train", columns=["date", "store_nbr"])
first_open = (sales.groupby("store_nbr")["date"].min()
              .rename("store_open_date").reset_index())
# store_open_date is the first date with a sales row: a proxy for the opening date, not an official date

cal = cal.drop(columns=["store_open_date", "is_open"], errors="ignore").merge(first_open, on="store_nbr", how="left")
cal["is_open"] = cal["date"] >= cal["store_open_date"]
cal["is_event_other"] = cal["is_event"] & ~cal["earthquake"]

miss = cal["transactions"].isna()
print("store-days before first sale:", int((~cal["is_open"]).sum()))
print("no transactions, before first sale:", int((miss & ~cal["is_open"]).sum()))
print("no transactions, store open:", int((miss & cal["is_open"]).sum()))
print(cal[miss & cal["is_open"]].groupby("date").size().sort_values(ascending=False).head(10))

cal.to_parquet(DIM / "calendar_store.parquet", index=False)

store-days before first sale: 6729
no transactions, before first sale: 6729
no transactions, store open: 719
date
2016-01-03    53
2016-01-01    53
2017-01-01    52
2015-01-01    47
2014-01-01    45
2016-01-04    39
2016-01-02    17
2016-09-27     4
2013-06-19     4
2016-09-11     2
dtype: int64


In [19]:
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
DIM = BASE / "clean" / "dim"

cal = pd.read_parquet(DIM / "calendar_store.parquet")
n_before = len(cal)

sales_days = (pd.read_parquet(BASE / "interim" / "train", columns=["date", "store_nbr"])
              .drop_duplicates())
sales_days["has_sales"] = True

cal = cal.drop(columns=["has_sales", "is_closed", "tx_gap"], errors="ignore")
cal = cal.merge(sales_days, on=["date", "store_nbr"], how="left")
cal["has_sales"] = cal["has_sales"].eq(True)
cal["is_closed"] = cal["is_open"] & ~cal["has_sales"]
cal["tx_gap"] = cal["transactions"].isna() & cal["has_sales"]

assert len(cal) == n_before, "merge changed the number of rows"
print("store-days:", f"{len(cal):,}")
print("closed (open store, no sales):", int(cal["is_closed"].sum()))
print("transactions gap (sales but no tx):", int(cal["tx_gap"].sum()))

cal.to_parquet(DIM / "calendar_store.parquet", index=False)
print("saved calendar_store with", cal.shape[1], "columns")

store-days: 90,936
closed (open store, no sales): 601
transactions gap (sales but no tx): 118
saved calendar_store with 26 columns


### Closure evidence: how strong is "suspected closure"?

**Purpose.** A store-day with no sales record is treated as a suspected closure (column `is_closed`). This step checks how well that is supported and separates strong from weak cases.

**Method.** (1) Compare two independent sources: open store-days with no sales row and, among them, those that also have no transactions row. (2) Classify each no-record day: *chain_wide* (at least 20 stores without records on the same date), *extended_run* (at least 7 consecutive days for one store) or *isolated*. (3) Count how many fall on a national holiday or 1 January.

**Output.** Counts per type and a new column `closure_type` in `calendar_store`.

**Limitation.** A missing record is evidence of a closure, not proof: a reporting outage looks identical, and isolated days are the weakest case. In the 10-store subset all of these days fall in the train period, so validation and test metrics are not affected; the printed check shows whether this also holds for all 54 stores.

In [20]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
DIM = BASE / "clean" / "dim"
CHAIN_MIN_STORES = 20      # this many stores without records on the same date = chain-wide event
LONG_RUN_DAYS = 7          # this many consecutive days for one store = extended gap

cal = pd.read_parquet(DIM / "calendar_store.parquet")
cal = cal.sort_values(["store_nbr", "date"]).reset_index(drop=True)

# 1) do the two independent sources (sales and transactions) agree?
open_no_sales = cal["is_open"] & ~cal["has_sales"]
print("open store-days with no sales row            :", int(open_no_sales.sum()))
print("... of which also no transactions row        :", int((open_no_sales & cal["transactions"].isna()).sum()))
print("... of which a transactions row exists       :", int((open_no_sales & cal["transactions"].notna()).sum()))

# 2) how strong is the evidence for each case?
closed = cal["is_closed"]
n_same_day = closed.groupby(cal["date"]).transform("sum")
grp = (~closed).groupby(cal["store_nbr"]).cumsum()
run_len = closed.groupby([cal["store_nbr"], grp]).transform("sum").where(closed, 0)
cal["closure_type"] = np.select(
    [~closed, n_same_day >= CHAIN_MIN_STORES, run_len >= LONG_RUN_DAYS],
    ["open", "chain_wide", "extended_run"], default="isolated")

cc = cal[closed]
print("\nno-record store-days by type:")
print(cc["closure_type"].value_counts().to_string())
on_holiday = cc["hol_national"].astype(bool) | (cc["date"].dt.strftime("%m-%d") == "01-01")
print(f"\non a national holiday or 1 January: {int(on_holiday.sum())} of {len(cc)}")
print("\nisolated cases, most frequent dates:")
print(cc.loc[cc["closure_type"] == "isolated", "date"].value_counts().head(5).to_string())

late = cc[cc["date"] >= "2017-07-15"]
print(f"\nno-record days in the validation/test period (all 54 stores): {len(late)}")

cal.to_parquet(DIM / "calendar_store.parquet", index=False)
print("\nsaved calendar_store with closure_type")

open store-days with no sales row            : 601
... of which also no transactions row        : 601
... of which a transactions row exists       : 0

no-record store-days by type:
closure_type
extended_run    398
chain_wide      196
isolated          7

on a national holiday or 1 January: 210 of 601

isolated cases, most frequent dates:
date
2015-07-07    1
2016-03-22    1
2016-03-23    1
2016-11-14    1
2016-11-15    1

no-record days in the validation/test period (all 54 stores): 0

saved calendar_store with closure_type


**Closure evidence – interpretation.** All 601 open store-days without a sales row also have no transactions row, and none has a transactions row without sales, so the two sources agree. Two thirds (398) belong to runs of at least 7 consecutive days for the same store, one third (196) are chain-wide events on 1 January, and only 7 are isolated days. This pattern fits temporary closures or reporting outages rather than random missing records. The data cannot tell a closure from an outage, so these days are treated as not observed (missing in lags and rolling features) rather than as zero demand. For all 54 stores, none of these days falls in the validation or test period (15 July to 15 August 2017), so evaluation does not depend on how they are treated.

## Data preparation log

| Step | Problem found | What was done | Why | Rows before → after |
|---|---|---|---|---|
| 1 | `train.csv` is 4.7 GB with 125,497,040 rows | Read in chunks of 10M rows with compact data types and saved as Parquet (notebook 01) | Lower memory (2.4 GB) and faster later steps | 125,497,040 → 125,497,040 |
| 2 | 7,795 negative sales (0.006%) | `unit_sales_raw` kept, `unit_sales` clipped at 0, `is_return` flag | Returns are not demand; very large negatives distort statistics | unchanged |
| 3 | `onpromotion` missing for 21,657,651 rows (17.3%), all of Jan 2013-Mar 2014 | Kept as unknown (-1), not filled with "no promotion" | Missing by structure, not at random | unchanged |
| 4 | 25 December missing in 2013-2016 | Not filled | Stores were closed; filling would distort the weekly pattern | unchanged |
| 5 | Sparse sales: no row with zero sales | Daily grid from each pair's first sale | Needed for lags, rolling features and ADI; no zeros before a product is sold | 125,497,040 → 194,660,451 |
| 6 | 601 open store-days with no sales and no transactions row (398 extended runs, 196 chain-wide, 7 isolated) | Flag `is_closed` and `closure_type`; missing in historical features; excluded from training and metrics | A closure cannot be told from a reporting outage; not zero demand | 52,605 rows in the 10-store subset |
| 7 | 6.53% of sales are not whole numbers (weighed products) | Decimals kept | Rounding loses information | unchanged |
| 8 | Extreme values (for example meats: 99.9th percentile 364, maximum 89,440) | Flagged later (`is_extreme`, notebook 03), not deleted | Errors cannot be told from bulk sales | unchanged |
| 9 | 64 of 4,100 items never sold | Dropped from the item table | They cannot be forecast | 4,100 → 4,036 items |
| 10 | Oil price missing on 521 of 1,684 days | Forward fill, first day left missing, `oil_imputed` flag | Last known price; no future information | 1,684 days |
| 11 | 12 transferred holidays; 38 dates with several events | Transferred holidays excluded; flags by scope; one row per date and store | Avoids double counting | calendar: 90,936 rows |
| 12 | 7,448 store-days without a transactions row | Kept missing; flags `is_open`, `has_sales`, `is_closed`, `tx_gap` | 6,729 before first sale, 601 no-record days, 118 gaps | unchanged |
| 13 | Duplicates of the key (date, store, item) | None found | The key is unique | unchanged |